In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

In [2]:
spark = (
    SparkSession.builder.appName("Rental Price Prediction - Proximity Features")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# --- Schools schema, full column set (matches dv402-SchoolLocations2025.csv) ---
schools_schema = StructType([
    StructField("Education_Sector",      StringType(), True),
    StructField("Entity_Type",           StringType(), True),
    StructField("School_No",             StringType(), True),
    StructField("School_Name",           StringType(), True),
    StructField("School_Type",           StringType(), True),
    StructField("School_Status",         StringType(), True),
    StructField("Address_Line_1",        StringType(), True),
    StructField("Address_Line_2",        StringType(), True),
    StructField("Address_Town",          StringType(), True),
    StructField("Address_State",         StringType(), True),
    StructField("Address_Postcode",      IntegerType(), True),
    StructField("Postal_Address_Line_1", StringType(), True),
    StructField("Postal_Address_Line_2", StringType(), True),
    StructField("Postal_Town",           StringType(), True),
    StructField("Postal_State",          StringType(), True),
    StructField("Postal_Postcode",       IntegerType(), True),
    StructField("Full_Phone_No",         StringType(), True),
    StructField("Region",                StringType(), True),
    StructField("Area",                  StringType(), True),
    StructField("LGA_ID",                IntegerType(), True),
    StructField("LGA_Name",              StringType(), True),
    StructField("LGA_TYPE",              StringType(), True),
    StructField("X",                     DoubleType(), True),  # longitude
    StructField("Y",                     DoubleType(), True),  # latitude
])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/25 17:58:16 WARN Utils: Your hostname, DESKTOP-9UAA9AK, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/25 17:58:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/25 17:58:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
SCHOOL_DATA_PATH = "../../external_data/dv402-SchoolLocations2025.csv"
df_schools = spark.read.csv(SCHOOL_DATA_PATH, header=True, schema=schools_schema)

In [4]:
df_schools.printSchema()

root
 |-- Education_Sector: string (nullable = true)
 |-- Entity_Type: string (nullable = true)
 |-- School_No: string (nullable = true)
 |-- School_Name: string (nullable = true)
 |-- School_Type: string (nullable = true)
 |-- School_Status: string (nullable = true)
 |-- Address_Line_1: string (nullable = true)
 |-- Address_Line_2: string (nullable = true)
 |-- Address_Town: string (nullable = true)
 |-- Address_State: string (nullable = true)
 |-- Address_Postcode: integer (nullable = true)
 |-- Postal_Address_Line_1: string (nullable = true)
 |-- Postal_Address_Line_2: string (nullable = true)
 |-- Postal_Town: string (nullable = true)
 |-- Postal_State: string (nullable = true)
 |-- Postal_Postcode: integer (nullable = true)
 |-- Full_Phone_No: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- LGA_ID: integer (nullable = true)
 |-- LGA_Name: string (nullable = true)
 |-- LGA_TYPE: string (nullable = true)
 |-- X: double (nullabl

In [5]:
df_schools.show(5, truncate=False)

+----------------+-----------+---------+-----------------------------+-----------+-------------+-------------------------+--------------+--------------+-------------+----------------+---------------------+---------------------+--------------+------------+---------------+-------------+----------------------+-----------------------+------+----------------+---------+----------+----------+
|Education_Sector|Entity_Type|School_No|School_Name                  |School_Type|School_Status|Address_Line_1           |Address_Line_2|Address_Town  |Address_State|Address_Postcode|Postal_Address_Line_1|Postal_Address_Line_2|Postal_Town   |Postal_State|Postal_Postcode|Full_Phone_No|Region                |Area                   |LGA_ID|LGA_Name        |LGA_TYPE |X         |Y         |
+----------------+-----------+---------+-----------------------------+-----------+-------------+-------------------------+--------------+--------------+-------------+----------------+---------------------+-----------------

### Create spacial location features

In [6]:
# --- Only keep open schools with valid coordinates ---
df_schools_open = df_schools.filter(F.col("School_Status") == "O")

df_schools_geo_raw = df_schools_open.select(
    "School_No",
    "School_Name",
    "School_Type",
    F.col("X").alias("school_lon"),
    F.col("Y").alias("school_lat"),
)

df_schools_geo = df_schools_geo_raw.filter(
    F.col("school_lon").isNotNull() & F.col("school_lat").isNotNull()
)

n_before = df_schools_geo_raw.count()
n_after = df_schools_geo.count()
n_dropped = n_before - n_after
print(f"Open schools before null-coordinate filter: {n_before}")
print(f"Dropped for missing lat/lon: {n_dropped}")
print(f"Remaining after filter: {n_after}")
if n_dropped > 0:
    print("Rows dropped:")
    df_schools_geo_raw.filter(
        F.col("school_lon").isNull() | F.col("school_lat").isNull()
    ).show(truncate=False)

schools_pd = df_schools_geo.toPandas()
schools_pd["School_Type"] = schools_pd["School_Type"].str.strip()

# Also flag any null/blank School_Type, since those rows silently match
# none of the feature_groups below (.isin() on NaN is False).
n_null_type = schools_pd["School_Type"].isna().sum()
print(f"Schools with null School_Type (will be excluded from all groups): {n_null_type}")

# Pri/Sec schools offer both levels, so they contribute to both the
# "primary" and "secondary" nearest-distance groups, in addition to
# their own dedicated group.
feature_groups = {
    "primary":   ["Primary", "Pri/Sec"],
    "secondary": ["Secondary", "Pri/Sec"],
    "pri_sec":   ["Pri/Sec"],
    "special":   ["Special"],
    "language":  ["Language"],
}

print("\nFeature groups:")
for name, types in feature_groups.items():
    n = schools_pd["School_Type"].isin(types).sum()
    print(f"  nearest_{name}_school  <-  {types}  ({n} schools)")

Open schools before null-coordinate filter: 2297
Dropped for missing lat/lon: 0
Remaining after filter: 2297
Schools with null School_Type (will be excluded from all groups): 0

Feature groups:
  nearest_primary_school  <-  ['Primary', 'Pri/Sec']  (1815 schools)
  nearest_secondary_school  <-  ['Secondary', 'Pri/Sec']  (608 schools)
  nearest_pri_sec_school  <-  ['Pri/Sec']  (248 schools)
  nearest_special_school  <-  ['Special']  (117 schools)
  nearest_language_school  <-  ['Language']  (5 schools)


### Verify school coordinates

Check `school_lat`/`school_lon` for duplicates and values outside
Victoria's bounding box, the same checks used for the other facility
datasets.

In [7]:
VIC_LAT_RANGE = (-39.3, -33.9)
VIC_LON_RANGE = (140.9, 150.1)

n_duplicate_coordinates = schools_pd.duplicated(
    subset=["school_lat", "school_lon"]
).sum()
print(f"Duplicate coordinate pairs: {n_duplicate_coordinates}")

is_in_victoria = (
    schools_pd["school_lat"].between(*VIC_LAT_RANGE)
    & schools_pd["school_lon"].between(*VIC_LON_RANGE)
)
n_outside = (~is_in_victoria).sum()
print(f"Outside Victoria: {n_outside}")

schools_pd[~is_in_victoria][
    ["School_No", "School_Name", "school_lat", "school_lon"]
]

Duplicate coordinate pairs: 0
Outside Victoria: 0


,School_No,School_Name,school_lat,school_lon


In [8]:
df_schools_geo_raw = df_schools_open.select(
    "Education_Sector",
    "School_No",
    "School_Name",
    "School_Type",
    F.col("X").alias("school_lon"),
    F.col("Y").alias("school_lat"),
)

df_schools_geo = df_schools_geo_raw.filter(
    F.col("school_lon").isNotNull() & F.col("school_lat").isNotNull()
)

# Export to CSV format

In [9]:
SCHOOLS_EXPORT_PATH = "../../external_data/schools.csv"
SCHOOLS_EXPORT_COLUMNS = [
    "School_No",
    "School_Name",
    "School_Type",
    "school_lat",
    "school_lon",
]

schools_export = schools_pd[SCHOOLS_EXPORT_COLUMNS]

print(f"School_No unique: {schools_export['School_No'].is_unique}")
print(schools_export.isna().sum())

schools_export.to_csv(SCHOOLS_EXPORT_PATH, index=False)
schools_export.head()

School_No unique: False
School_No      0
School_Name    0
School_Type    0
school_lat     0
school_lon     0
dtype: int64


,School_No,School_Name,School_Type,school_lat,school_lon
0,20,Parade College,Secondary,-37.690178,145.066978
1,25,Simonds Catholic College,Secondary,-37.805971,144.952883
2,26,St Mary’s College Melbourne,Secondary,-37.859365,144.997001
3,28,St Patrick's College Ballarat,Secondary,-37.559711,143.831558
4,29,St Patrick's School,Primary,-37.564397,143.847147
